# [LAB-04] 데이터 재구조화

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 피벗 테이블 실습 데이터 가져오기

In [2]:
origin = load_data("city_people")
origin

📚 서울, 인천, 부산에서 2005년, 2010년 2015년에 조사한 가상의 인구수 데이터(인덱스와 메타데이터 없음)


,도시,연도,인구,지역
0,서울,2015,9904312,수도권
1,서울,2010,9631482,수도권
2,서울,2005,9762546,수도권
3,부산,2015,3448737,경상권
4,부산,2010,3393191,경상권
5,부산,2005,3512547,경상권
6,인천,2015,2890451,수도권
7,인천,2010,2632035,수도권


## #02. 피벗 테이블

### 1. 기본 사용 방법 (도시와 연도에 따른 인구수 재배치)

- 인덱스, 컬럼, 값으로 사용할 필드를 각각 지정하여 데이터를 재배치한다.
- 리턴되는 피벗 테이블 역시 데이터 프레임 객체이다.

In [3]:
pivot1 = pivot_table(origin,
                     index= '도시',
                     columns= '연도', 
                     values= '인구')
pivot1

연도,2005,2010,2015
도시,,,
부산,3512547.000,3393191.000,3448737.000
서울,9762546.000,9631482.000,9904312.000
인천,NaN,2632035.000,2890451.000


### 2. 중복 데이터의 집계 방법 지정하기

- 인덱스와 컬럼에 따른 value가 두 개 이상인 경우 집계 방법을 지정해야 정확한 결과를 얻을 수 있다.
  - 예제에서는 2010년과 2015년에 대한 수도권 데이터가 두 개씩 존재한다.
  - 전체 인구수를 얻어야 하는 경우라면 합계를 구해야 한다.

In [4]:
pivot2 = pivot_table(origin,
                     index='지역',
                     columns='연도',
                     values='인구'
                     ,aggfunc='sum')
pivot2

연도,2005,2010,2015
지역,,,
경상권,3512547,3393191,3448737
수도권,9762546,12263517,12794763


### 3. 두 개 이상의 집계 함수 지정

- 집계 함수의 이름을 리스트로 설정한다.

In [6]:
pivot3 = pivot_table(origin,
                     index='지역',
                     columns='연도',
                     values='인구',
                     aggfunc=['sum', 'mean'])
pivot3

sum                            mean                        
연도      2005      2010      2015        2005        2010        2015
지역                                                                  
경상권  3512547   3393191   3448737 3512547.000 3393191.000 3448737.000
수도권  9762546  12263517  12794763 9762546.000 6131758.500 6397381.500

### 4. 복수 인덱스 지정

In [7]:
pivot4 = pivot_table(origin,
                     index=['지역', '연도'],
                     columns='도시',
                     values='인구',
                     aggfunc=['mean', 'sum'])
pivot4

mean                                 sum              \
도시                부산          서울          인천          부산          서울   
지역  연도                                                                 
경상권 2005 3512547.000         NaN         NaN 3512547.000         NaN   
    2010 3393191.000         NaN         NaN 3393191.000         NaN   
    2015 3448737.000         NaN         NaN 3448737.000         NaN   
수도권 2005         NaN 9762546.000         NaN         NaN 9762546.000   
    2010         NaN 9631482.000 2632035.000         NaN 9631482.000   
    2015         NaN 9904312.000 2890451.000         NaN 9904312.000   

                      
도시                인천  
지역  연도                
경상권 2005         NaN  
    2010         NaN  
    2015         NaN  
수도권 2005         NaN  
    2010 2632035.000  
    2015 2890451.000

## #03. melt

### 1. 샘플 피벗 테이블 생성

In [9]:
pivot_df = pivot_table(origin,
                       index='연도', columns='지역',
                       values='인구', aggfunc='mean')
pivot_df

지역,경상권,수도권
연도,,
2005,3512547.000,9762546.000
2010,3393191.000,6131758.500
2015,3448737.000,6397381.500


### 2. 피벗 테이블 분리 1단계

- 데이터프레임의 인덱스를 일반 컬럼으로 설정한다.

In [10]:
pivot_df2 = pivot_df.reset_index()
pivot_df2

지역,연도,경상권,수도권
0,2005,3512547.000,9762546.000
1,2010,3393191.000,6131758.500
2,2015,3448737.000,6397381.500


### 3. 피벗 테이블 분리 2단계 : melt 처리

- id_vars : 인덱스로 사용할 컬럼이름. 반드시 컬럼만 가능(인덱스 불가)
- value_vars: 분리할 컬럼 이름들

In [11]:
melt1 = melt(pivot_df2, id_vars=['연도'],
             value_vars=['경상권', '수도권'])
melt1

,연도,지역,value
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


### 4. 피벗 테이블 분리시 필드 이름 지정하기

In [12]:
melt2 = melt(pivot_df2, id_vars=['연도'],
             value_vars=['경상권', '수도권'],
             var_name='구분', value_name='인구수')
melt2

,연도,구분,인구수
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


## #04. Stack, Unstack 실습을 위한 준비작업

### 1. 샘플 데이터 가져오기

In [13]:
origin = load_data("body_size")
origin

📚 어느 학급 학생들의 이름, 성별, 키, 몸무게에 대한 가상의 데이터

    field    description
--  -------  -------------
 0  name     이름
 1  sex      성별
 2  height   키
 3  weight   몸무게



,sex,height,weight
name,,,
Lee,M,175,98.000
Park,F,167,48.000
Hong,M,180,NaN
Kim,F,162,55.000
Nam,M,172,85.000


## #05. stack

### 1. 기본 사용 방법

- 인덱스 열에 따라 모든 변수를 하나의 변수로 쌓아놓는 처리로 리턴 결과는 Series 객체가 된다.
- 원본 데이터프레임의 컬럼이름이 카테고리 역할을 하는 변수로 배치되고 이에 따라 원본의 값이 재배치 된다.
- **빈 값(`NaN`)은 stack 과정에서 제거된다.**
  - 예제에서는 `Hong`의 `Weight`가 제외되어 있음

In [14]:
stack1 = origin.stack()
print(type(stack1))
stack1

<class 'pandas.Series'>


name        
Lee   sex           M
      height      175
      weight   98.000
Park  sex           F
      height      167
      weight   48.000
Hong  sex           M
      height      180
      weight      NaN
Kim   sex           F
      height      162
      weight   55.000
Nam   sex           M
      height      172
      weight   85.000
dtype: object

### 2. Stack 결과를 DataFrame으로 생성하기

In [15]:
df1 = DataFrame(origin.stack())
df1

0
name              
Lee  sex         M
     height    175
     weight 98.000
Park sex         F
     height    167
     weight 48.000
Hong sex         M
     height    180
     weight    NaN
Kim  sex         F
     height    162
     weight 55.000
Nam  sex         M
     height    172
     weight 85.000

## #06. Unstack

### 1. stack 결과를 원래대로 되돌린다.

In [16]:
df1.unstack()

0              
     sex height weight
name                  
Lee    M    175 98.000
Park   F    167 48.000
Hong   M    180    NaN
Kim    F    162 55.000
Nam    M    172 85.000

In [ ]:
# 연습문제 1.
# 1. 학생 흡연율 데이터 전처리
# students_smoke 데이터 셋은 한 도시의 20개 중,고교를 대상으로 조사한 흡연율 자료.
# 도시와 농촌별에 따른 남여 평균 흡연율을 확인할 수 있는 데이터 피벗테이블로 구성하시요.

In [19]:
# 0. 데이터 불러오기
data = load_data('students_smoke')
data.info()
data

📚 한 도시의 20개 중,고교를 대상으로 조사한 흡연율 자료 (인덱스와 메타데이터 없음)
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지역      20 non-null     str    
 1   구성      20 non-null     str    
 2   흡연율     20 non-null     float64
dtypes: float64(1), str(2)
memory usage: 804.0 bytes


,지역,구성,흡연율
0,도시,남,0.640
1,도시,여,0.450
2,농촌,공학,0.700
3,도시,남,0.850
4,농촌,남,0.720
5,도시,남,0.780
6,도시,여,0.620
7,도시,남,0.790
8,농촌,남,0.750
9,농촌,공학,0.810


In [20]:
# 1. 도시와 농촌별 남여 평균 피벗테이블로 구성
data1 = pivot_table(data,
                    index= '지역',
                    columns='구성',
                    values= '흡연율',
                    aggfunc= 'mean')
data1

구성,공학,남,여
지역,,,
농촌,0.757,0.745,0.463
도시,0.680,0.792,0.507


In [ ]:
#연습문제 2. 동네 아이스크림 가게 매출 분석
# 사장님은 여름 시즌을 맞아 어떤 맛 아이스크림이 인기가 많은지, 날짜별 판매 추이는 어떤지 궁금해하며, 효율적인 재고 관리를 위한 데이터 분석을 의뢰
# icecream_sales 데이터셋을 사용하여 다음의 의뢰를 완수하세요.
# 1. 데이터의 전체적인 구조와 정보를 확인하고, 결측치가 있는지 확인하세요. 만약 결측치가 있다면, 해당 행을 제거
# 2. 판매 날짜를 인덱스로, 아이스크림 맛을 컬럼으로 하여 각 맛의 일별 판매량을 보여주는 피벗테이블을 만드세요
# 3. 사장님이 다른 분석 툴에서 사용하고 싶다며, 위에서 만들 피벗 테이블을 다시 긴 형식(Long format)의 데이터로 분리해 다라고 요청했습니다. 컬럼 이름은 원본 이름 그대로 유지

In [29]:
# 0. 데이터 불러오기
ice = load_data('icecream_sales')
ice.info()
ice

📚 '달콤 스쿱' 아이스크림 가게의 매출 데이터 (인덱스 없음)
<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      12 non-null     datetime64[us]
 1   Flavor    12 non-null     str           
 2   Topping   11 non-null     str           
 3   Price     12 non-null     int64         
 4   Quantity  11 non-null     float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 818.0 bytes


,Date,Flavor,Topping,Price,Quantity
0,2023-07-01,초콜릿,아몬드,3500,20.000
1,2023-07-01,바닐라,초코시럽,3000,25.000
2,2023-07-01,딸기,연유,3200,18.000
3,2023-07-02,민트초코,초코칩,3800,15.000
4,2023-07-02,초콜릿,아몬드,3500,22.000
5,2023-07-02,바닐라,NaN,3000,30.000
6,2023-07-03,딸기,연유,3200,25.000
7,2023-07-03,민트초코,초코칩,3800,NaN
8,2023-07-03,바닐라,초코시럽,3000,28.000
9,2023-07-04,초콜릿,아몬드,3500,18.000


In [31]:
# 1. 결측치 확인
ice.isnull()
ice2 = ice.dropna(ignore_index=True)
ice2

,Date,Flavor,Topping,Price,Quantity
0,2023-07-01,초콜릿,아몬드,3500,20.000
1,2023-07-01,바닐라,초코시럽,3000,25.000
2,2023-07-01,딸기,연유,3200,18.000
3,2023-07-02,민트초코,초코칩,3800,15.000
4,2023-07-02,초콜릿,아몬드,3500,22.000
5,2023-07-03,딸기,연유,3200,25.000
6,2023-07-03,바닐라,초코시럽,3000,28.000
7,2023-07-04,초콜릿,아몬드,3500,18.000
8,2023-07-04,딸기,연유,3200,22.000
9,2023-07-04,민트초코,초코칩,3800,17.000


In [33]:
pivot_ice = pivot_table(ice2,
                        index='Date',
                        columns='Flavor',
                        values='Quantity',)
pivot_ice

Flavor,딸기,민트초코,바닐라,초콜릿
Date,,,,
2023-07-01,18.000,NaN,25.000,20.000
2023-07-02,NaN,15.000,NaN,22.000
2023-07-03,25.000,NaN,28.000,NaN
2023-07-04,22.000,17.000,NaN,18.000


In [38]:
melt_ice = pivot_ice.reset_index()
m_ice = melt(melt_ice, id_vars=['Date'],
                value_vars=['딸기', '민트초코', '바닐라', '초콜릿'])
m_ice

,Date,Flavor,value
0,2023-07-01,딸기,18.000
1,2023-07-02,딸기,NaN
2,2023-07-03,딸기,25.000
3,2023-07-04,딸기,22.000
4,2023-07-01,민트초코,NaN
5,2023-07-02,민트초코,15.000
6,2023-07-03,민트초코,NaN
7,2023-07-04,민트초코,17.000
8,2023-07-01,바닐라,25.000
9,2023-07-02,바닐라,NaN


In [ ]:
# 연습문제 3.
# game_scores 데이터 셋은 최근 열린 게임 토너먼트의 경기 결과 데이터이다.
# 데이터는 선수 이름('player')과 각 스테이지('stage1', 'stage2', 'stage3')점수로 구성되어있습니다.
# 일부 선수는 특정 스테이지에 참여하지 않아 점수가 비어있을 수 있습니다.
# 모든 스테이지에 참여한 선수들 중에서, 평균점수가 가장 높은 "최고의 선수"를 찾아내고, 그 선수의 평균 점수를 계산

In [ ]:
# 0. 준비단계
game = load_data('game_scores')
game.info()
game

📚 최근 열린 게임 토너먼트 경기 결과 데이터 (인덱스와 메타데이터 없음)
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   player  5 non-null      str    
 1   stage1  5 non-null      int64  
 2   stage2  4 non-null      float64
 3   stage3  4 non-null      float64
dtypes: float64(2), int64(1), str(1)
memory usage: 315.0 bytes


,player,stage1,stage2,stage3
0,Alice,85,92.000,88.000
1,Bob,78,NaN,95.000
2,Charlie,90,88.000,91.000
3,David,82,85.000,NaN
4,Eva,95,98.000,99.000


In [53]:
# 1. 각 스테이지 점수가 열로 구성된 데이터를, 선수와 스테이지 레벨에 따른 점수를 나타내는 형태로 재구조화하세요.
game1 = game.dropna()

game2 = game1.set_index('player')
game2

game3 = DataFrame(game2.stack())
game3

0
player               
Alice   stage1 85.000
        stage2 92.000
        stage3 88.000
Charlie stage1 90.000
        stage2 88.000
        stage3 91.000
Eva     stage1 95.000
        stage2 98.000
        stage3 99.000

In [60]:
# 2. 선수별 평균 점수 계산
game4 = game3.reset_index()
game4.filter(['player',0]).groupby('player').mean()

,0
player,
Alice,88.333
Charlie,89.667
Eva,97.333


### 선수별 평균 점수 계산하는거 다시해볼것